In [1]:
!nvidia-smi

Sat Aug  8 11:49:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             36W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q transformers datasets peft trl bitsandbytes accelerate huggingface_hub

In [2]:
from huggingface_hub import login
login()

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded.")
print(f"Memory used: {model.get_memory_footprint() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model loaded.
Memory used: 2.20 GB


In [4]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,175,040 || all params: 3,221,924,864 || trainable%: 0.2848


In [5]:
from datasets import load_dataset

# Load your training dataset
train_dataset = load_dataset('json', data_files='/kaggle/input/datasets/navag10/train-dataset/train_formatted.json', split='train')

# Load your holdout dataset (replace 'holdout_filename.json' with your actual file name)
holdout_dataset = load_dataset('json', data_files='/kaggle/input/datasets/navag10/holdout-dataset/holdout_formatted.json', split='train')

# Now you can print them without an error!
print(train_dataset)
print(holdout_dataset)


Dataset({
    features: ['text'],
    num_rows: 8000
})
Dataset({
    features: ['text'],
    num_rows: 1000
})


In [ ]:
!ls -R /kaggle/working


In [ ]:
!pip install -U trl==0.12.0


In [ ]:
!pip install -U "trl<0.14.0"


In [8]:
from trl import SFTTrainer, SFTConfig

# Move parameters back inside SFTConfig
training_args = SFTConfig(
    output_dir="/kaggle/working/checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=25,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    report_to="none",
    
    # Text mapping parameters go here 
    dataset_text_field="text",
    max_seq_length=1024 # Note: Use 'max_seq_length' instead of 'max_length'
)

# Initialize the Trainer cleanly
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=training_args
)

print("Trainer ready.")


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Trainer ready.


In [10]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/kaggle/working/checkpoints",
    
    # --- SPEED OPTIMIZATION STRATEGY ---
    num_train_epochs=1,               # Cut the training loops down to 1 single epoch
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,    # UPDATE SHIFT: Disabling accumulation removes compilation loops 
    learning_rate=2e-4,
    bf16=True,                        # Retain native 16-bit precision matrix operations
    
    # --- UI & MANAGEMENT SPEEDUPS ---
    logging_steps=5,                  # Fast log refreshes to check real-time loss tracking
    save_strategy="no",               # Stop saving heavy intermediate checkpoints during the run
    report_to="none",
    
    # --- TEXT MAPPING PARAMETERS ---
    dataset_text_field="text",
    max_seq_length=512                # SQUASHING MATRICES: Cutting sequence length from 1024 halves memory load
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=training_args
)

print("Optimized high-speed trainer ready.")


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Optimized high-speed trainer ready.


In [ ]:
# Start training
trainer.train()

# Save final adapter to kaggle
trainer.save_model("/kaggle/working/final_adapter")
tokenizer.save_pretrained("/kaggle/working/final_adapter")

print("Training complete. Adapter saved.")